# Analyze Scores

## Notebook Preferences

In [ ]:
%load_ext autoreload
%autoreload 2
%config InlineBackend.figure_format="retina"

## Libraries

In [ ]:
import buckaroo  # noqa: F401
import lamindb as ln
import spatialdata as sd
import pandas as pd
import matplotlib.pyplot as plt
import nbl
from upath import UPath
import seaborn as sns
import seaborn.objects as so
from matplotlib.patches import Patch
from collections.abc import Sequence
import itertools
from tqdm.auto import tqdm
from great_tables import GT, md
import numpy as np
import pymupdf
from PIL import Image

## Configuration

In [ ]:
pd.set_option("future.no_silent_downcasting", True)
pd.set_option("future.infer_string", False)
pd.set_option("mode.copy_on_write", False)

In [ ]:
# cluster = DaskLocalCluster(n_workers=10, threads_per_worker=1)
# cluster(open_dashboard=True)

In [ ]:
from seaborn import axes_style

theme_dict = {**axes_style("whitegrid"), "grid.linestyle": ":", "axes.facecolor": "w", "axes.edgecolor": "slategray"}

so.Plot.config.theme.update(theme_dict)

In [ ]:
figures_upath = UPath("../../../../data/db/figures/scoring/")
figures_upath.mkdir(exist_ok=True, parents=True)

In [ ]:
ln.settings.sync_git_repo = "https://github.com/karadavis-lab/nbl.git"
ln.track("q0As4ij1QA2A0000")

## Load Data

In [ ]:
nbl_sdata = sd.read_zarr(ln.Artifact.get(description="NBL Tissue Samples", is_latest=True).path)

In [ ]:
ratio_scores = [
    "ratio",
    "ratio_no_TH",
]
normalized_difference_scores = [
    "normalized_difference",
    "normalized_difference_no_TH",
]
log_ratio_scores = [
    "log_ratio",
    "log_ratio_no_TH",
]
scaled_difference_scores = [
    "scaled_difference",
    "scaled_difference_no_TH",
]
all_scores = ratio_scores + normalized_difference_scores + log_ratio_scores + scaled_difference_scores

In [ ]:
score_data = nbl_sdata.tables["nbl_wc_diagnosis"].obs[["Risk", *all_scores]]

In [ ]:
def add_fig_legend(fig: plt.Figure, n_groups: int, group_labels: Sequence[str], ncol: int, loc: str = "lower center"):  # noqa: D103
    colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]
    handles = []
    for group_idx in range(n_groups):
        color = colors[group_idx]
        handles.append(Patch(edgecolor=color, facecolor=color, fill=False))
    fig.legend(handles=handles, labels=group_labels, loc=loc, ncol=ncol)
    fig.subplots_adjust(top=0.9, left=0.1, right=0.9, bottom=0.1)
    return fig

In [ ]:
from scipy.stats import ks_2samp, bws_test, anderson_ksamp, PermutationMethod

In [ ]:
from collections.abc import Mapping, Sequence
from typing import Any
from pydantic import BaseModel


class Risk(BaseModel):  # noqa: D101
    risk_A: str
    risk_B: str


class ScoreStats(BaseModel):  # noqa: D101
    stat_method: str
    score_info: Mapping[str, str] | Sequence[str]
    risks: Risk
    result: Any

In [ ]:
risks = ["Low", "Intermediate", "High"]

In [ ]:
ks_2_scores = []
for risk_combos in itertools.combinations(risks, 2):
    risk_A, risk_B = risk_combos
    risk_A_df = score_data.query(f"Risk == '{risk_A}'")
    risk_B_df = score_data.query(f"Risk == '{risk_B}'")
    ks_2 = ks_2samp(risk_A_df[all_scores], risk_B_df[all_scores])
    ks_2_scores.append(
        ScoreStats(
            stat_method="two-sample Kolmogorov-Smirnov",
            score_info=all_scores,
            risks=Risk(risk_A=risk_A, risk_B=risk_B),
            result=ks_2,
        )
    )

In [ ]:
[k.result.pvalue for k in ks_2_scores]

In [ ]:
[k.result.pvalue for k in ks_2_scores]

In [ ]:
# Calculate the total number of iterations
total_iterations = len(list(itertools.combinations(risks, 2))) * len(all_scores)

bws_scores = []
with tqdm(total=total_iterations, desc="Processing risk combinations and scores") as pbar:
    for risk_combos in itertools.combinations(risks, 2):
        risk_A, risk_B = risk_combos
        risk_A_df = score_data.query(f"Risk == '{risk_A}'")
        risk_B_df = score_data.query(f"Risk == '{risk_B}'")
        for score in all_scores:
            bws = bws_test(risk_A_df[score], risk_B_df[score])
            bws_scores.append(
                ScoreStats(
                    stat_method="Baumgartner-Weiss-Schindler",
                    score_info=[score],
                    risks=Risk(risk_A=risk_A, risk_B=risk_B),
                    result=bws,
                )
            )
            pbar.update(1)

In [ ]:
anderson_scores = []
with tqdm(total=total_iterations) as pbar:
    for risk_combos in list(itertools.combinations(risks, 2)):
        risk_A, risk_B = risk_combos
        risk_A_df = score_data.query(f"Risk == '{risk_A}'")
        risk_B_df = score_data.query(f"Risk == '{risk_B}'")
        for score in all_scores:
            aksamp = anderson_ksamp(
                samples=[risk_A_df[score], risk_B_df[score]], method=PermutationMethod(n_resamples=100, batch=10)
            )
            anderson_scores.append(
                ScoreStats(
                    stat_method="Anderson-Darling K Sample",
                    score_info=[score],
                    risks=Risk(risk_A=risk_A, risk_B=risk_B),
                    result=aksamp,
                ),
            )
            pbar.update(1)

In [ ]:
import pandas as pd


# Convert the list of ScoreStats objects to a DataFrame with one row per score
import pandas as pd


def score_stats_to_dataframe(score_stats_list):  # noqa: D103
    data = []
    for score_stat in score_stats_list:
        # Ensure score_info and pvalue are iterable
        pvalue_list = (
            [score_stat.result.pvalue] if isinstance(score_stat.result.pvalue, float) else score_stat.result.pvalue
        )

        # Process each pair of score_info and pvalue
        for score, pval in zip(score_stat.score_info, pvalue_list, strict=False):
            data.append(
                {
                    "stat_method": score_stat.stat_method,
                    "score": score,
                    "risk_A": score_stat.risks.risk_A,
                    "risk_B": score_stat.risks.risk_B,
                    "pvalue": pval,
                }
            )
    return pd.DataFrame(data)


ks_2_scores_df = score_stats_to_dataframe(ks_2_scores)
bws_scores_df = score_stats_to_dataframe(bws_scores)
anderson_ksamp_df = score_stats_to_dataframe(anderson_scores)

In [ ]:
def set_pvalue_star(x: pd.Series):  # noqa: D103
    if x > 0.05:
        return "ns"
    if x <= 0.05 and x > 0.01:
        return "✩"
    if x <= 0.01 and x > 0.001:
        return "✩✩"
    if x <= 0.001 and x > 0.0001:
        return "✩✩✩"
    if x <= 0.0001:
        return "✩✩✩✩"

In [ ]:
scores_df = pd.concat([anderson_ksamp_df, bws_scores_df, ks_2_scores_df]).sort_values(["stat_method", "score"])

scores_df = scores_df.rename(columns={"score": "Score", "stat_method": "Statistic", "pvalue": "P-value"})
scores_df["P-value Stars"] = scores_df["P-value"].apply(set_pvalue_star)

In [ ]:
scores_df = scores_df.reset_index(drop=True)

In [ ]:
scores_df = pd.read_csv("scores.csv", index_col=0)

In [ ]:
gts = []
for g in scores_df.groupby("Score"):
    gt = (
        GT(g[1].drop(columns=["Score"]), rowname_col="Statistic")
        .tab_header(title="Scores Analysis Table", subtitle=f"All NBL Diagnosis Cells | {g[0]}")
        .tab_stubhead("Statistic")
        .tab_spanner(label=md("**Risks**"), columns=["risk_A", "risk_B"])
        .tab_spanner(label=md("**P Values**"), columns=["P-value", "P-value Stars"])
        .fmt_scientific(columns="P-value", n_sigfig=4)
    )
    gts.append(gt)
    gt.save(f"./scores_{g[0]}.pdf", web_driver="safari", scale=4, encoding="utf-16")

In [ ]:
fig = plt.figure(figsize=(16, 10), dpi=300)
subfigs = fig.subfigures(nrows=2, ncols=2)
subfigs2 = []
for sf in subfigs.flat:
    subfigs2.append(sf.subfigures(nrows=1, ncols=2, wspace=0.01))

boxen_plot_ax = []
for sf in subfigs2:
    sf[0].subplots()
    sf[1].subplots(nrows=2, ncols=1, sharex=True, sharey=True, gridspec_kw={"hspace": 0.05})

for boxen_plot_subfig, score_group in zip(
    subfigs2, [ratio_scores, log_ratio_scores, normalized_difference_scores, scaled_difference_scores], strict=False
):
    boxen_plot_ax = boxen_plot_subfig[0].axes[0]
    sns.boxenplot(
        data=score_data.melt(
            id_vars=["Risk"],
            value_vars=score_group,
            value_name="score",
        ),
        x="variable",
        y="score",
        hue="Risk",
        fill=False,
        legend=None,
        ax=boxen_plot_ax,
    )
    boxen_plot_ax.set_xlabel("")
    boxen_plot_ax.set_ylabel("Score Value")

    for table_ax, score in zip(boxen_plot_subfig[1].axes, score_group, strict=False):
        d = pymupdf.open(f"./scores_{score}.pdf")
        pix = list(d.pages())[0].get_pixmap(alpha=False)
        img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
        table_ax.imshow(img, aspect="auto")
        nbl.util.remove_ticks(table_ax, axis="xy")
fig.suptitle(t="Diagnosis Samples NBL Scores")

fig = add_fig_legend(fig, n_groups=3, group_labels=["High", "Intermediate", "Low"], ncol=2, loc="lower center")
fig.patch.set_facecolor("white")

fig_path: UPath = figures_upath / "scores_boxenplot.pdf"
fig.savefig(fig_path)
artifact = ln.Artifact(
    data=fig_path,
    description="Scores Boxenplot -- No Downsampling",
)
artifact.save()

In [ ]:
from warnings import warn
from sklearn.neighbors import KDTree
from numpydantic import NDArray


def uniform_downsampling(data: pd.DataFrame, sample_size: int | float, **kwargs):  # noqa: D103
    if isinstance(sample_size, int):
        if sample_size >= data.shape[0]:
            warn(
                f"Number of observations larger than requested sample size {sample_size}, "
                f"returning complete data (n={data.shape[0]})",
                stacklevel=2,
            )
            return data
        return data.sample(n=sample_size, **kwargs)
    if isinstance(sample_size, float):
        return data.sample(frac=sample_size, **kwargs)
    raise TypeError("sample_size should be an int or float value")


def prob_downsample(local_d: NDArray, target_d: int, outlier_d: int) -> NDArray:  # noqa: D103
    result = np.zeros(local_d.shape, dtype=float)
    # Condition 1: local_d <= outlier_d -> return 0
    result[local_d <= outlier_d] = 0
    # Condition 2: outlier_d < local_d <= target_d -> return 1
    condition2 = (outlier_d < local_d) & (local_d <= target_d)
    result[condition2] = 1
    # Condition 3: local_d > target_d -> return target_d / local_d
    condition3 = local_d > target_d
    result[condition3] = target_d / local_d[condition3]

    return result


def density_probability_assignment(  # noqa: D103
    sample: pd.DataFrame,
    data: pd.DataFrame,
    distance_metric: str = "manhattan",
    alpha: int = 5,
    outlier_dens: int = 1,
    target_dens: int = 5,
) -> NDArray:
    tree = KDTree(sample, metric=distance_metric, leaf_size=100)
    dist, _ = tree.query(data, k=2)
    dist = np.median([x[1] for x in dist])
    dist_threshold = dist * alpha
    ld = tree.query_radius(data, r=dist_threshold, count_only=True)
    od = np.percentile(ld, q=outlier_dens)
    td = np.percentile(ld, q=target_dens)
    prob: NDArray = np.apply_along_axis(prob_downsample, axis=0, arr=ld, target_d=td, outlier_d=od)
    return prob


def density_dependent_downsampling(  # noqa: D103
    data: pd.DataFrame,
    features: list = None,
    sample_size: int | float = 0.1,
    alpha: int = 5,
    distance_metric: str = "manhattan",
    tree_sample_size: int | float = 0.1,
    outlier_dens: int = 1,
    target_dens: int = 5,
):
    if isinstance(sample_size, int) and sample_size >= data.shape[0]:
        warn("Requested sample size >= size of dataframe", stacklevel=2)
        return data
    df = data.copy()  # noqa: PD901
    features = features or df.columns.tolist()
    tree_sample = uniform_downsampling(data=df, sample_size=tree_sample_size)
    prob = density_probability_assignment(
        sample=tree_sample[features],
        data=df[features],
        distance_metric=distance_metric,
        alpha=alpha,
        outlier_dens=outlier_dens,
        target_dens=target_dens,
    )
    if sum(prob) == 0:
        warn(
            "Error: density dependendent downsampling failed; weights sum to zero. " "Defaulting to uniform sampling",
            stacklevel=2,
        )
        return uniform_downsampling(data=data, sample_size=sample_size)
    if isinstance(sample_size, int):
        return df.sample(n=sample_size, weights=prob)
    return df.sample(frac=sample_size, weights=prob)

In [ ]:
downsampled_score_data = density_dependent_downsampling(
    data=score_data,
    features=all_scores,
    sample_size=0.1,
)

In [ ]:
ks_2_ddd_scores = []
for risk_combos in itertools.combinations(risks, 2):
    risk_A, risk_B = risk_combos
    risk_A_df = downsampled_score_data.query(f"Risk == '{risk_A}'")
    risk_B_df = downsampled_score_data.query(f"Risk == '{risk_B}'")
    ks_2 = ks_2samp(risk_A_df[all_scores], risk_B_df[all_scores])
    ks_2_ddd_scores.append(
        ScoreStats(
            stat_method="two-sample Kolmogorov-Smirnov",
            score_info=all_scores,
            risks=Risk(risk_A=risk_A, risk_B=risk_B),
            result=ks_2,
        )
    )

In [ ]:
# Calculate the total number of iterations
total_iterations = len(list(itertools.combinations(risks, 2))) * len(all_scores)

bws_ddd_scores = []
with tqdm(total=total_iterations, desc="Processing risk combinations and scores") as pbar:
    for risk_combos in itertools.combinations(risks, 2):
        risk_A, risk_B = risk_combos
        risk_A_df = downsampled_score_data.query(f"Risk == '{risk_A}'")
        risk_B_df = downsampled_score_data.query(f"Risk == '{risk_B}'")
        for score in all_scores:
            bws = bws_test(risk_A_df[score], risk_B_df[score])
            bws_ddd_scores.append(
                ScoreStats(
                    stat_method="Baumgartner-Weiss-Schindler",
                    score_info=[score],
                    risks=Risk(risk_A=risk_A, risk_B=risk_B),
                    result=bws,
                )
            )
            pbar.update(1)

In [ ]:
anderson_ddd_scores = []
with tqdm(total=total_iterations) as pbar:
    for risk_combos in list(itertools.combinations(risks, 2)):
        risk_A, risk_B = risk_combos
        risk_A_df = downsampled_score_data.query(f"Risk == '{risk_A}'")
        risk_B_df = downsampled_score_data.query(f"Risk == '{risk_B}'")
        for score in all_scores:
            aksamp = anderson_ksamp(
                samples=[risk_A_df[score], risk_B_df[score]], method=PermutationMethod(n_resamples=100, batch=10)
            )
            anderson_ddd_scores.append(
                ScoreStats(
                    stat_method="Anderson-Darling K Sample",
                    score_info=[score],
                    risks=Risk(risk_A=risk_A, risk_B=risk_B),
                    result=aksamp,
                ),
            )
            pbar.update(1)

In [ ]:
ks_2_ddd_scores_df = score_stats_to_dataframe(ks_2_ddd_scores)
bws_ddd_scores_df = score_stats_to_dataframe(bws_ddd_scores)
anderson_ddd_ksamp_df = score_stats_to_dataframe(anderson_ddd_scores)

In [ ]:
scores_ddd_df = pd.concat([anderson_ddd_ksamp_df, bws_ddd_scores_df, ks_2_ddd_scores_df]).sort_values(
    ["stat_method", "score"]
)

scores_ddd_df = scores_ddd_df.rename(columns={"score": "Score", "stat_method": "Statistic", "pvalue": "P-value"})
scores_ddd_df["P-value Stars"] = scores_ddd_df["P-value"].apply(set_pvalue_star)
scores_ddd_df = scores_ddd_df.reset_index(drop=True)

In [ ]:
gts = []
for g in scores_ddd_df.groupby("Score"):
    gt = (
        GT(g[1].drop(columns=["Score"]), rowname_col="Statistic")
        .tab_header(title="Scores Analysis Table", subtitle=f"All NBL Diagnosis Cells | {g[0]}")
        .tab_stubhead("Statistic")
        .tab_spanner(label=md("**Risks**"), columns=["risk_A", "risk_B"])
        .tab_spanner(label=md("**P Values**"), columns=["P-value", "P-value Stars"])
        .fmt_scientific(columns="P-value", n_sigfig=4)
    )
    gts.append(gt)
    gt.save(f"./scores_ddd_{g[0]}.pdf", web_driver="safari", scale=4, encoding="utf-16")

In [ ]:
theme_dict = {"grid.linestyle": ":", "axes.facecolor": "w", "axes.edgecolor": "slategray"}

sns.set_theme(style="whitegrid", font_scale=0.75, rc=theme_dict)

In [ ]:
fig = plt.figure(figsize=(16, 10), dpi=300)
subfigs = fig.subfigures(nrows=2, ncols=2)
subfigs2 = []
for sf in subfigs.flat:
    subfigs2.append(sf.subfigures(nrows=1, ncols=2, wspace=0.01))

boxen_plot_ax = []
for sf in subfigs2:
    sf[0].subplots()
    sf[1].subplots(nrows=2, ncols=1, sharex=True, sharey=True, gridspec_kw={"hspace": 0.05})

for boxen_plot_subfig, score_group in zip(
    subfigs2, [ratio_scores, log_ratio_scores, normalized_difference_scores, scaled_difference_scores], strict=False
):
    boxen_plot_ax = boxen_plot_subfig[0].axes[0]
    sns.boxenplot(
        data=downsampled_score_data.melt(
            id_vars=["Risk"],
            value_vars=score_group,
            value_name="score",
        ),
        x="variable",
        y="score",
        hue="Risk",
        fill=False,
        legend=None,
        ax=boxen_plot_ax,
    )
    boxen_plot_ax.set_xlabel("")
    boxen_plot_ax.set_ylabel("Score Value")

    for table_ax, score in zip(boxen_plot_subfig[1].axes, score_group, strict=False):
        d = pymupdf.open(f"./scores_ddd_{score}.pdf")
        pix = list(d.pages())[0].get_pixmap(alpha=False)
        img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
        table_ax.imshow(img, aspect="auto")
        nbl.util.remove_ticks(table_ax, axis="xy")
fig.suptitle(t="Density Downsampled Diagnosis Samples NBL Scores")
fig = add_fig_legend(fig, n_groups=3, group_labels=["High", "Intermediate", "Low"], ncol=2, loc="lower center")
fig.patch.set_facecolor("white")

fig_path: UPath = figures_upath / "density_scores_boxenplot.pdf"
fig.savefig(fig_path)
artifact = ln.Artifact(
    data=fig_path,
    description="Scores Boxenplot -- Density Downsampled",
)
artifact.save()

In [ ]:
ln.finish()